# LangChain Tutorial (Step by Step) using Grok

## Requirements
For this notebook, you need:
1. `langchain`, `langchain-groq`, and `python-dotenv`.
2. A free Groq account and an API key.
3. `GROQ_API_KEY` available from `.env` or from a secure prompt.

Optional in PowerShell:
`$env:GROQ_API_KEY="your_api_key_here"`

In [25]:
%pip install -U langchain langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import os
import getpass
import importlib

# Load .env if python-dotenv is available
dotenv_spec = importlib.util.find_spec("dotenv")
if dotenv_spec is not None:
    dotenv_module = importlib.import_module("dotenv")
    dotenv_module.load_dotenv()

def _clean_key(value: str | None) -> str:
    if not value:
        return ""
    return value.strip().strip("\"").strip("'")

# Ask for the key only if it is still missing
if not os.getenv("GROQ_API_KEY"):
    try:
        entered_key = getpass.getpass("Enter GROQ_API_KEY: ")
    except Exception:
        entered_key = input("Enter GROQ_API_KEY (visible): ")
    os.environ["GROQ_API_KEY"] = _clean_key(entered_key)

if not _clean_key(os.getenv("GROQ_API_KEY")):
    raise EnvironmentError("GROQ_API_KEY not found. Add it to .env or enter it when prompted.")

print("GROQ_API_KEY detected")

GROQ_API_KEY detected


## 1) Initialize the model
Create a chat model instance that will be reused across the tutorial.

In [27]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

print("Model initialized: llama-3.1-8b-instant")

Model initialized: llama-3.1-8b-instant


## 2) Direct invocation with messages
Send a system instruction and user message directly to the model.

This step shows the most basic way to call a chat model in LangChain.

In [28]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a helpful weather assistant. Provide brief, clear responses."),
    HumanMessage(content="What's the weather like in San Francisco today?"),
]

response = model.invoke(messages)
print(response.content)

I'm not aware of the current date. However, I can suggest checking a reliable weather website or app for the most up-to-date information. If you'd like, I can provide general information about San Francisco's typical weather patterns.


## 3) Add an output parser
A parser normalizes model output into a plain string for easier downstream use.

In [29]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
chain_with_parser = model | parser

parsed_text = chain_with_parser.invoke(messages)
print(parsed_text)

I'm not aware of the current date. However, I can suggest checking a reliable weather website or app for the most up-to-date information. If you'd like, I can provide general information about San Francisco's typical weather patterns.


## 4) Build prompt templates
Prompt templates let you parameterize instructions (for language, input text, style, etc.).

In [30]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions about {topic}."),
    ("user", "{question}"),
])

prompt_value = template.invoke({"topic": "weather", "question": "How is the temperature measured?"})
for message in prompt_value.messages:
    print(f"[{message.__class__.__name__}] {message.content}")

[SystemMessage] You are a helpful assistant that answers questions about weather.
[HumanMessage] How is the temperature measured?


## 5) Compose the full chain
Combine `template | model | parser` for a reusable translation pipeline.

In [31]:
full_chain = template | model | parser

examples = [
    {"topic": "weather", "question": "What causes rain?"},
    {"topic": "geography", "question": "What is the capital of France?"},
]

for item in examples:
    out = full_chain.invoke(item)
    print(f"Q: {item['question']}")
    print(f"A: {out}\n")

Q: What causes rain?
A: Rain is a vital part of the Earth's water cycle, and it's caused by a combination of atmospheric and geographical factors. Here's a simplified explanation:

1. **Evaporation**: The first step in the process is evaporation. When the sun heats the surface of the Earth, it causes water to evaporate from oceans, lakes, rivers, and even plants. This water vapor rises into the air as gas.
2. **Condensation**: As the water vapor rises, it cools down, and its temperature decreases. When it reaches its dew point, the water vapor condenses into tiny droplets, forming **clouds**.
3. **Nucleation**: The condensed water droplets in the clouds need a surface to condense onto, known as a nucleus. This can be a dust particle, salt crystal, or even an airplane contrail. The droplets then grow and merge with other droplets, forming larger droplets.
4. **Precipitation**: When the droplets in the cloud become too heavy to remain suspended in the air, they fall to the ground as **pr

## 6) Stream model output
Streaming is useful for chat-like UX because tokens appear as they are generated.

In [32]:
print("Streaming response:\n")
for chunk in full_chain.stream({"topic": "climate", "question": "Why is the sky blue?"}):
    print(chunk, end="", flush=True)
print()

Streaming response:

The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.

Here's what happens: when sunlight enters Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2). These molecules are much smaller than the wavelength of light, so they scatter the light in all directions.

Now, here's the important part: shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths. This is because the smaller molecules are more effective at scattering the shorter wavelengths. As a result, the blue light is distributed throughout the atmosphere, giving the sky its blue appearance.

Think of it like this: imagine you're shining a flashlight through a prism. The different colors of light will bend at slightly different angles, and the blue light will bend the most. That's basically what's happening with the sky - the 

## Next step
From here you can adapt the same pattern to a RAG pipeline by replacing the user text with retrieved context + question.

In [33]:
print("Tutorial completed ")

Tutorial completed 


In [34]:
# Optional quick check
full_chain.invoke({"topic": "weather", "question": "What's the difference between weather and climate?"})

'Weather and climate are two related but distinct concepts in the field of meteorology.\n\n**Weather** refers to the short-term and local conditions of the atmosphere at a specific place and time, including temperature, humidity, cloudiness, wind, precipitation, and other factors. Weather is what you experience on a daily basis, such as a sunny day, a thunderstorm, or a heatwave. Weather is constantly changing and can vary significantly from one location to another.\n\n**Climate**, on the other hand, refers to the long-term average atmospheric conditions in a particular region, including temperature, precipitation, and other factors. Climate is a broader concept that describes the typical weather patterns of a region over a long period of time, usually 30 years or more. Climate is influenced by a combination of factors, including latitude, altitude, ocean currents, and the Earth\'s rotation.\n\nTo illustrate the difference, consider this example:\n\n* Weather: "It\'s going to be sunny 